# L2C — Local PDF-to-JSON-to-report pipeline

This notebook explores the dataset, optionally benchmarks local OCR, and executes extraction, annotation association, Annex A validation, reconciliation and PDF generation on one project. Automatic association has partial coverage; unresolved cases remain visible.

Confidential data and results stay outside Git/OneDrive. Launch with jupyter_server_config.py so saved outputs remain empty.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
assert (REPO / "l2c").is_dir(), "Launch from the repository root or notebooks directory."
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
DATA_DIR = Path(r"C:\Users\Admin\Downloads\l2c-participants")
OUTPUT_DIR = Path(r"C:\Users\Admin\Documents\Codex\l2c-local\notebook")
PROJECT = "CLP"
PAGE = 4
RUN_ML = False
assert DATA_DIR.is_dir(), "Set DATA_DIR to the local dataset."
for path in (DATA_DIR.resolve(), OUTPUT_DIR.resolve()):
    assert not path.is_relative_to(REPO), "Keep data outside the repository."
    assert not any("onedrive" in part.lower() for part in path.parts), "Use a folder outside OneDrive."
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Configuration ready. ML enabled:", RUN_ML)


## 1. Explore the dataset

Inventory page counts and spreadsheet dimensions. This checks readability, not extraction accuracy.


In [ ]:
from l2c.inspect import inspect_dataset
inventory = inspect_dataset(DATA_DIR)
(OUTPUT_DIR / "inventory.json").write_text(json.dumps(inventory, indent=2), encoding="utf-8")
for project in inventory["projects"]:
    errors = sum("error_type" in f for f in project["files"])
    print(project["project"], "PDFs:", project["pdf_count"], "pages:", project["page_count"], "read errors:", errors)
print("Spreadsheets:", len(inventory["spreadsheets"]))


## 2. Prepare identical image crops

Three crops are selected around native bar-designation candidates. This biases the test toward text-bearing regions. Native text stays local for comparison; the models receive image crops only. These are not validated Annex A element records.


In [ ]:
PLAN = DATA_DIR / PROJECT / f"L2C_PLAN_STR_{PROJECT}.pdf"
SAMPLES_DIR = OUTPUT_DIR / PROJECT / "samples"
subprocess.run([sys.executable, "-m", "l2c.prepare_samples", "--pdf", str(PLAN),
                "--page", str(PAGE), "--output-dir", str(SAMPLES_DIR), "--count", "3"],
               cwd=REPO, check=True)
samples = json.loads((SAMPLES_DIR / "samples.json").read_text(encoding="utf-8"))
print("Prepared crops:", len(samples))


## 3. Optional local ML inference

Install requirements-ml.txt and CPU PyTorch first. Set RUN_ML=True to execute. Florence requires downloaded model weights. PaddleOCR downloads public weights if the cache is empty; image inference stays local. Qwen was not run on the tested computer because available memory was insufficient for the proposed configuration.


In [ ]:
ML_PYTHON = Path(os.environ.get("L2C_ML_PYTHON", sys.executable))
BACKENDS = tuple(b.strip() for b in os.environ.get("L2C_ML_BACKENDS", "paddle").split(",") if b.strip())
MODEL_DIR = Path(os.environ.get("L2C_FLORENCE_MODEL_DIR", str(OUTPUT_DIR / "models" / "florence")))
model_env = os.environ.copy()
model_env.update({
    "HF_HOME": str(OUTPUT_DIR / "hf-cache"),
    "HF_HUB_DISABLE_TELEMETRY": "1",
    "PADDLE_PDX_CACHE_HOME": str(OUTPUT_DIR / "paddle-cache"),
    "PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK": "True",
    "OMP_NUM_THREADS": "1",
})
OCR_MODELS = os.environ.get("L2C_OCR_MODELS_DIR")
if RUN_ML:
    assert OCR_MODELS, "Set L2C_OCR_MODELS_DIR to preloaded Paddle weights."
    assert ML_PYTHON.is_file(), "Set L2C_ML_PYTHON to your ML interpreter."
    for backend in BACKENDS:
        assert backend in ("paddle", "florence"), "Unknown backend."
        command = [str(ML_PYTHON), "-m", "l2c.ml_benchmark", "--backend", backend,
                   "--samples", str(SAMPLES_DIR / "samples.json"),
                   "--output", str(SAMPLES_DIR / f"{backend}-results.json")]
        if backend == "paddle":
            command += ["--model-dir", OCR_MODELS]
        if backend == "florence":
            assert MODEL_DIR.is_dir(), "Set L2C_FLORENCE_MODEL_DIR to local weights."
            command += ["--model-dir", str(MODEL_DIR)]
        print(f"Running {backend} locally...", flush=True)
        subprocess.run(command, cwd=REPO, env=model_env, check=True)
    print("Selected models finished.")
else:
    print("ML skipped. Set RUN_ML=True after installing dependencies and weights.")


## 4. Compare saved outputs without displaying confidential transcripts

Metrics count distinct values per crop against native PDF text. They are not manually verified accuracy and do not evaluate element association or discrepancies.


In [ ]:
from l2c.ml_benchmark import tokens, quantity_pairs
for backend in ("paddle", "florence"):
    result_path = SAMPLES_DIR / f"{backend}-results.json"
    if not result_path.exists():
        print(backend, ": no inference results yet")
        continue
    report = json.loads(result_path.read_text(encoding="utf-8"))
    bar_matches = bar_total = pair_matches = pair_total = 0
    durations = []
    for sample, result in zip(samples, report["results"]):
        native = " ".join(sample["native_text"])
        a, b = tokens(native), tokens(result["text"])
        qa, qb = quantity_pairs(native), quantity_pairs(result["text"])
        bar_matches += len(a & b); bar_total += len(a)
        pair_matches += len(qa & qb); pair_total += len(qa)
        durations.append(result["seconds"])
    print(backend, "bar matches:", f"{bar_matches}/{bar_total}",
          "quantity-bar matches:", f"{pair_matches}/{pair_total}",
          "mean seconds:", round(sum(durations) / len(durations), 3))


## 5. Run the complete project pipeline

Default: native PDF extraction on CLP. Set RUN_ML=True and L2C_OCR_MODELS_DIR to preloaded Paddle weights for hybrid OCR verification. L2C_CONTEXT_FILE may point to local reviewer regions; manual context is declared in association evidence. The PDF lists coverage and unresolved annotations.


In [ ]:
from l2c.pipeline import run_project
PIPELINE_DIR = OUTPUT_DIR / PROJECT / "pipeline"
OCR_MODELS = os.environ.get("L2C_OCR_MODELS_DIR")
REVIEW_CONTEXT = os.environ.get("L2C_CONTEXT_FILE")
OCR_BUDGET = os.environ.get("L2C_MAX_OCR_PAGES")
if RUN_ML:
    assert OCR_MODELS, "Set L2C_OCR_MODELS_DIR to preloaded local Paddle weights."
    command = [str(ML_PYTHON), "-m", "l2c.pipeline", "--project-dir", str(DATA_DIR / PROJECT),
               "--output-dir", str(PIPELINE_DIR), "--ocr", "hybrid", "--models-dir", OCR_MODELS]
    if OCR_BUDGET is not None:
        command += ["--max-ocr-pages", OCR_BUDGET]
    if REVIEW_CONTEXT:
        command += ["--context", REVIEW_CONTEXT]
    subprocess.run(command, cwd=REPO, env=model_env, check=True)
else:
    run_project(DATA_DIR / PROJECT, PIPELINE_DIR, ocr="off", context_file=REVIEW_CONTEXT)
summary = json.loads((PIPELINE_DIR / "run-summary.json").read_text(encoding="utf8"))
assert not summary["errors"], "Inspect run-summary.json for page errors."
print({k: summary[k] for k in ("project", "processed_pages", "record_count", "unresolved_annotations", "ocr_verified_crops", "ocr_processed_pages", "ocr_skipped_pages", "seconds")})


## 6. Validate and locate deliverables

Annex A stays separate from evidence and reconciliation. The reported coverage is essential: a PDF with no detected discrepancy does not establish conformity.


In [ ]:
from l2c.annex_a import Annotation
records = json.loads((PIPELINE_DIR / "annex-a.json").read_text(encoding="utf8"))
for record in records:
    Annotation.model_validate(record)
assert len({r["id"] for r in records}) == len(records)
assert (PIPELINE_DIR / "report.pdf").is_file()
print("Validated Annex A records:", len(records))
print("Deliverables folder:", PIPELINE_DIR)
print("Partial-coverage prototype; remaining associations require review.")
